In [2]:
# First we got to do some imports
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import time

from sklearn.linear_model import LinearRegression


from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.pipeline import Pipeline

import contextlib, io
from itertools import combinations
import shap


# Random seed for reproducibility
np.random.seed(42)

all_targetv = ["tradeflow_baci", "tradeflow_comtrade_o", "tradeflow_comtrade_d", 
                "tradeflow_imf_o", "tradeflow_imf_d", "combined_trade_baci",]

df = pd.read_csv("data/ecowas_df_full_lag_zero.csv")

train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}
required_columnsv2 = {"gdp_o", "gdp_d", "dist"}

def baseline_loglinear(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016):
    '''
    The main function for creating our log-linear regression baseline
    
    Input:
        df = main pandas DataFrame to work with.
        target_variable = str of the column which we are inferring for. 
        columns = list of column elements that are included in the model. NOTE: Always include the basic Gravity columns, other we raise an error
        year_split = int of year where we split between train and test. Defaults to 2016
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    
    all_needed = columns + [target_variable]

    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = LinearRegression()
    model.fit(X_train, y_train)


    # Finally we can get some workable metrics from this regression:
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample R²:", r2)


    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                                cov_kwds={"groups": groups},)

    print(ols.summary())


    return {
        "model_name": target_variable,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }

    
def main(random_state:int = 42):
    '''
    
    '''

    np.random.seed(random_state)


c:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\bachelor_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# TO IMPLEMENT:

# SHAP summary for the variables in the ML models, to see which parameter has the most weight. 


legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic", "comlang_ethno", "comlang_off", "contig"}

conflict_groups = {
    "disorder":    [c for c in df.columns if c.startswith("disorder_")],
    "event":       [c for c in df.columns if c.startswith("event_")],
    "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
    "target":      [c for c in df.columns if c.startswith("target_")],
}

base_groups = list(conflict_groups.keys())

In [4]:
def multilayer_perceptron(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016, set_random_state=42,
    hidden_layer=(100, 50),  alpha=0.0001, learning_rate_init=0.001):
    '''
    Function for applying multilayer perceptrons to the dataframe. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns = 
        year_split = 
        set_random_state = 
        hidden_layer = 
        alpha = 
        learning_rate_init =
    
    output:
        model = 
        dict containing various measures of fit.
    '''

    columns = columns.copy()

    # We do the standard checks for variables
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")
    
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")
    
    # Prepare the data frames
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()
    
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    # log the values for both train and test.
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Update the list to have the new log strings
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    
    # We encode the strings
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    model = Pipeline(
        steps=[ 
            ("scaler", StandardScaler()), 
            ("mlp", MLPRegressor(
                hidden_layer_sizes=hidden_layer, 
                activation="relu",  
                solver="adam", 
                alpha=alpha,
                learning_rate_init=learning_rate_init,
                max_iter=1000, 
                random_state=set_random_state
            )),
        ]
    )

    model.fit(X_train, y_train)

    # SHAP is a little more complicated for Pipelines
    def predict_fn(X):
        '''
        Predicting on the dataframe, called by the KernelExplainer. 
        Please note, veeeery slow, so run on subsets of the data
        '''
        return model.predict(pd.DataFrame(X, columns=X_train.columns))
    
    background = shap.sample(X_train, 100) # Define a smaller set of X_train to run the code on. Otherwise it will take -forever-
    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(X_train, silent=True)
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"MLP_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }


def xgboost_regression(df: pd.DataFrame, target_variable: str, columns: list, year_split: int = 2016, set_random_state: int = 42,
    n_estimators: int = 500, learning_rate: float = 0.05, max_depth: int = 6, subsample: float = 0.8, colsample_bytree: float = 0.8, min_child_weight: int = 1):
    '''
    Function for applying XGBoost to the dataframe. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns =
        year_split = 
        set_random_state = 
        n_estimators = 
        learning_rate = 
        max_depth = 
        subsample = 
        colsample_bytree = 
        min_child_weight = 
    
    output:
        model = 
        dict with various measures of fit.
    '''
    columns = columns.copy()

    # Same old quick validation checks
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")


    # Train/test split happens here
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()


    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables (for train/test)
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = pd.get_dummies(test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)
        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    # Then we can create our model
    model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        random_state=set_random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    # Final evaluation and scores - Here we also add the SHAP scores that will give us parameter weights!
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train) # These are SHAP values on all training data 
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value


    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"XGBoost_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }


def random_forest_regression(df: pd.DataFrame, target_variable: str, columns: list, year_split: int = 2016, set_random_state: int = 42,
    n_estimators: int = 500, max_depth: int | None = None, min_samples_leaf: int = 1, min_samples_split: int = 2, max_features: str | float = "sqrt"):
    '''
    Random Forest regression following the conventions and transformations of the baseline linear FE model. SHAP scores are implemented and exported with the dict.

    input:
        df = 
        target_variable = 
        columns = 
        year_split = 
        set_random_state = 
        n_estimators = 
        max_depth = 
        min_samples_leaf = 
        min_samples_split = 
        max_features = 

    output:
        dict with various measures of fit.
        model = 
    '''

    columns = columns.copy()

    # A few small validation checks, to make sure we have the right columns
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    # train/test split
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()


    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Replace raw gravity vars with log versions in columns list
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        # Align test columns to train (handles missing categories)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # -----------------------------
    # Final model matrices
    # -----------------------------
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]



    # we create the random forest model, using the input variables as defined by the function call
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
        max_features=max_features,
        random_state=set_random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # Final evaluation and scores - Here we also add the SHAP scores that will give us parameter weights!
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train) # These are SHAP values on all training data 
    shap_importance = np.abs(shap_values).mean(axis=0)
    shap_importance_dict = dict(zip(X_train.columns, shap_importance))
    grouped_shap = {"exporter_FE": 0.0, "importer_FE": 0.0, "other": {}} # We can combine the iso3_o and iso3_d to measure them as fixed effects
    for feature, value in shap_importance_dict.items():
        if feature.startswith("iso3_o_"):
            grouped_shap["exporter_FE"] += value
        elif feature.startswith("iso3_d_"):
            grouped_shap["importer_FE"] += value
        else:
            grouped_shap["other"][feature] = value

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    return model, {
        "model_name": f"RandomForest_{target_variable}",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "shap importance (grouped)": grouped_shap,
    }



def xgboost_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing xgboost performance with a varied combination of conflict inputs

    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = xgboost_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df



def random_forest_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing Random Forest performance with a varied combination of conflict inputs
    
    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = random_forest_regression(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df

def mlp_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    '''
    Function for comparing Multilayer Perceptron performance with a varied combination of conflict inputs

    input:

    
    '''
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    combos_to_test = []

    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))


    results = []
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = multilayer_perceptron(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)
    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — MLP {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [6]:
# We need to run on a subset of files.
df2 = pd.read_csv("data/synth/ecowas_df_synthetic_full_lag_zero_0.csv")

mlp_model, dict_mlp = multilayer_perceptron(df2, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"], hidden_layer = (42,), alpha = 0.01, learning_rate_init = 0.0001)


c:\Users\mhm25\Desktop\ITU\6thSemester\bachelorproj\bachelor_2026\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [8]:
rf_model, dict_rf = random_forest_regression(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])

In [7]:
dict_mlp

{'model_name': 'MLP_tradeflow_comtrade_d',
 'n_train': 3640,
 'n_test': 910,
 'rmse': 6.396421359806685,
 'mae': 3.072149344218009,
 'r2': -4.296422465913436,
 'shap importance (grouped)': {'exporter_FE': np.float64(1.0045544632688148),
  'importer_FE': np.float64(1.344285492506873),
  'other': {'contig': np.float64(0.10535620699939409),
   'comlang_off': np.float64(0.04998524949989689),
   'comlang_ethno': np.float64(0.03891701421415277),
   'disorder_Demonstrations': np.float64(0.002103237089227159),
   'disorder_Political violence': np.float64(0.012639117000388403),
   'disorder_Political violence; Demonstrations': np.float64(0.01124542071028838),
   'disorder_Strategic developments': np.float64(0.010126397877617366),
   'event_Battles': np.float64(0.02553087087871773),
   'event_Explosions/Remote violence': np.float64(0.007836605471631975),
   'event_Protests': np.float64(0.015316101569396019),
   'event_Riots': np.float64(0.025448684620031612),
   'event_Strategic developments': n

In [1]:
rf_basic_model, dict_rf_basic = random_forest_regression(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'], n_estimators=100)
rf_model, dict_rf = random_forest_regression(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"], n_estimators=100)

mlp_basic_model, dict_mlp_basic = multilayer_perceptron(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
mlp_model, dict_mlp = multilayer_perceptron(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])

xgboost_basic_model, dict_xgboost_basic = xgboost_regression(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
xgboost_model, dict_xgboost = xgboost_regression(df, "tradeflow_comtrade_d", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])


NameError: name 'random_forest_regression' is not defined

In [11]:
dict_xgboost

{'model_name': 'XGBoost_tradeflow_baci',
 'n_train': 2892,
 'n_test': 618,
 'rmse': 2.1296505371963206,
 'mae': 1.5065732975592918,
 'r2': 0.5968762601227564,
 'shap importance (grouped)': {'exporter_FE': np.float32(1.1101698),
  'importer_FE': np.float32(0.46810687),
  'other': {'contig': np.float32(0.4491746),
   'comlang_off': np.float32(0.10808191),
   'comlang_ethno': np.float32(0.032321386),
   'disorder_Demonstrations': np.float32(0.06595184),
   'disorder_Political violence': np.float32(0.055114746),
   'disorder_Political violence; Demonstrations': np.float32(0.01419752),
   'disorder_Strategic developments': np.float32(0.05263044),
   'event_Battles': np.float32(0.036723703),
   'event_Explosions/Remote violence': np.float32(0.027408522),
   'event_Protests': np.float32(0.043209143),
   'event_Riots': np.float32(0.032676153),
   'event_Strategic developments': np.float32(0.0075428463),
   'event_Violence against civilians': np.float32(0.029581858),
   'perpetrator_Civilians':

In [11]:
df2 = pd.read_csv("data/synth/ecowas_df_synthetic_full_lag_zero_0.csv")
df3 = pd.read_csv("data/synth/ecowas_df_synthetic_full_lag_one_0.csv")
df4 = pd.read_csv("data/synth/ecowas_df_synthetic_full_lag_two_0.csv")

df_list = [df2, df3, df4]
listing_variables = ["tradeflow_baci", "tradeflow_comtrade_o", "tradeflow_comtrade_d", "tradeflow_imf_o", "tradeflow_imf_d", "combined_trade_baci"]
text_holder = ["", "", ""]

for i in range(3):
    for j in range(6):
        rf_basic_model, dict_rf_basic = random_forest_regression(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'], n_estimators=100)
        rf_model, dict_rf = random_forest_regression(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"], n_estimators=100)

        mlp_basic_model, dict_mlp_basic = multilayer_perceptron(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
        mlp_model, dict_mlp = multilayer_perceptron(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])

        xgboost_basic_model, dict_xgboost_basic = xgboost_regression(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
        xgboost_model, dict_xgboost = xgboost_regression(df_list[i], listing_variables[j], columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'] + conflict_groups["disorder"] + conflict_groups["event"] + conflict_groups["perpetrator"] + conflict_groups["target"])

        text_holder[i] += (
            f"\nVariable: {listing_variables[j]}\n"
            f"RF basic: {dict_rf_basic}\n"
            f"RF full: {dict_rf}\n"
            f"MLP basic: {dict_mlp_basic}\n"
            f"MLP full: {dict_mlp}\n"
            f"XGB basic: {dict_xgboost_basic}\n"
            f"XGB full: {dict_xgboost}\n"
        )

        


In [12]:
text_holder

["\nVariable: tradeflow_baci\nRF basic: {'model_name': 'RandomForest_tradeflow_baci', 'n_train': 3640, 'n_test': 910, 'rmse': 2.1969234699638394, 'mae': 1.5707153376721754, 'r2': 0.48236444266050016, 'shap importance (grouped)': {'exporter_FE': np.float64(1.0825656624986466), 'importer_FE': np.float64(0.5285056627256377), 'other': {'contig': np.float64(0.3804110992038566), 'comlang_off': np.float64(0.10057214821250529), 'comlang_ethno': np.float64(0.062156051780437885), 'log_gdp_o': np.float64(0.7346613360759036), 'log_gdp_d': np.float64(0.4353649468565382), 'log_distw': np.float64(0.5444576907836274)}}}\nRF full: {'model_name': 'RandomForest_tradeflow_baci', 'n_train': 3640, 'n_test': 910, 'rmse': 2.347837909776248, 'mae': 1.7718221272556067, 'r2': 0.40880539026556406, 'shap importance (grouped)': {'exporter_FE': np.float64(0.5906058946141746), 'importer_FE': np.float64(0.4119161509340978), 'other': {'contig': np.float64(0.3599775106497096), 'comlang_off': np.float64(0.079590705716251

### RESULTS: ecowas_df_full_lag_zero

In [ ]:
print(dict_rf_basic)   # 100 estimators
print(dict_rf)
print(dict_mlp_basic)
print(dict_mlp)
print(dict_xgboost_basic)
print(dict_xgboost)

{'model_name': 'RandomForest_tradeflow_baci', 'n_train': 2892, 'n_test': 618, 'rmse': 1.841271505914709, 'mae': 1.2505564227087556, 'r2': 0.6986596066214918, 'shap importance (grouped)': {'exporter_FE': np.float64(1.4100028588274187), 'importer_FE': np.float64(0.5489301121252249), 'other': {'contig': np.float64(0.4904233794604901), 'comlang_off': np.float64(0.1370171587450892), 'comlang_ethno': np.float64(0.09553333640925231), 'log_gdp_o': np.float64(0.9198915347712162), 'log_gdp_d': np.float64(0.6099998574709274), 'log_distw': np.float64(0.6493084176197)}}}
{'model_name': 'RandomForest_tradeflow_baci', 'n_train': 2892, 'n_test': 618, 'rmse': 2.270328036365904, 'mae': 1.6565444984703928, 'r2': 0.5418592624419744, 'shap importance (grouped)': {'exporter_FE': np.float64(0.7644572541535598), 'importer_FE': np.float64(0.4159385183885952), 'other': {'contig': np.float64(0.4497059851179167), 'comlang_off': np.float64(0.12584729166593234), 'comlang_ethno': np.float64(0.06639101290313273), 'di

In [15]:
print(dict_rf_basic)   # 100 estimators
print(dict_rf)
print(dict_mlp_basic)
print(dict_mlp)
print(dict_xgboost_basic)
print(dict_xgboost)

{'model_name': 'RandomForest_tradeflow_comtrade_o', 'n_train': 2429, 'n_test': 476, 'rmse': 1.6719683914668257, 'mae': 1.1428609131315213, 'r2': 0.7333161885021535, 'shap importance (grouped)': {'exporter_FE': np.float64(1.3719783028933155), 'importer_FE': np.float64(0.48919210637474825), 'other': {'contig': np.float64(0.5807117929961289), 'comlang_off': np.float64(0.1581137639878064), 'comlang_ethno': np.float64(0.0993193148303064), 'log_gdp_o': np.float64(0.938866573575145), 'log_gdp_d': np.float64(0.6212154213324648), 'log_distw': np.float64(0.6598172892831768)}}}
{'model_name': 'RandomForest_tradeflow_comtrade_o', 'n_train': 2429, 'n_test': 476, 'rmse': 2.1795530573257995, 'mae': 1.5763318387796992, 'r2': 0.5468151123404343, 'shap importance (grouped)': {'exporter_FE': np.float64(0.8063560382110551), 'importer_FE': np.float64(0.39814131640451245), 'other': {'contig': np.float64(0.5518072658774064), 'comlang_off': np.float64(0.11014905662060766), 'comlang_ethno': np.float64(0.071972

In [5]:
print(dict_rf_basic)   # 100 estimators
print(dict_rf)
print(dict_mlp_basic)
print(dict_mlp)
print(dict_xgboost_basic)
print(dict_xgboost)

{'model_name': 'RandomForest_tradeflow_comtrade_d', 'n_train': 2471, 'n_test': 466, 'rmse': 1.8623889523544321, 'mae': 1.2525587135380998, 'r2': 0.6886332164653832, 'shap importance (grouped)': {'exporter_FE': np.float64(1.585311763591145), 'importer_FE': np.float64(0.6518617057037284), 'other': {'contig': np.float64(0.3162383834925066), 'comlang_off': np.float64(0.13114127982781248), 'comlang_ethno': np.float64(0.0728579445301022), 'log_gdp_o': np.float64(1.0446228091678116), 'log_gdp_d': np.float64(0.45406082161968603), 'log_distw': np.float64(0.5040033646824085)}}}
{'model_name': 'RandomForest_tradeflow_comtrade_d', 'n_train': 2471, 'n_test': 466, 'rmse': 2.400009892354139, 'mae': 1.795102406183005, 'r2': 0.4829202020445347, 'shap importance (grouped)': {'exporter_FE': np.float64(0.9132692349984615), 'importer_FE': np.float64(0.5241775003710714), 'other': {'contig': np.float64(0.293431852609045), 'comlang_off': np.float64(0.09931535988645868), 'comlang_ethno': np.float64(0.045744880